# Try GPSBabel

https://www.gpsbabel.org/htmldoc-1.9.0/gpsbabel-1.9.0.pdf

# Run in Bash unix shell

Do this if the ordering of gps data does not work in python 

#sort -t, -k2,2 large_gps.csv > large_gps_sorted.csv

In [1]:
import time
start = time.time()

import pandas as pd
import geopandas as gpd
import movingpandas as mpd
from shapely.geometry import Point
import datetime
from geopy.distance import geodesic 
import numpy as np
import fiona
import matplotlib.pyplot as plt
from datetime import timedelta
import seaborn as sns
#from joblib import Parallel, delayed

## 1) Load GPS data and sort by ID and time

In [2]:
test_data_gdf = gpd.read_file('Ferry_Road_Code/FerryRoadIntersect2019.csv')

# Convert longitude/latitude columns to geometry
test_data_gdf = gpd.GeoDataFrame(test_data_gdf,
    geometry=gpd.points_from_xy(test_data_gdf['Longitude of Visit'], test_data_gdf['Latitude of Visit']), 
    crs="EPSG:4326")

print(f'Loaded GPS data as a geodataframe')

test_data = None # Set to none for memory

test_data_gdf['t'] = pd.to_datetime(test_data_gdf['Date'] + ' ' + test_data_gdf['Time of Day'])
print(f'Made datetime column')

test_data_gdf = test_data_gdf.sort_values(by=['Hashed Device ID', 'Unix Timestamp of Visit'])
print(f'Sorted Dataframe by user ID and time')

print(test_data_gdf.head(n=5))

Loaded GPS data as a geodataframe
Made datetime column
Sorted Dataframe by user ID and time
      field_1 Polygon ID                          Hashed Device ID  \
95472  581196        NZL  000350f4f44b025bf9c0477fc51ad88426b170bb   
95473  581197        NZL  000350f4f44b025bf9c0477fc51ad88426b170bb   
95474  581198        NZL  000350f4f44b025bf9c0477fc51ad88426b170bb   
95475  581199        NZL  000350f4f44b025bf9c0477fc51ad88426b170bb   
94519  557331        NZL  000350f4f44b025bf9c0477fc51ad88426b170bb   

      Latitude of Visit Longitude of Visit Unix Timestamp of Visit  \
95472         -43.54204          172.66379              1569453724   
95473         -43.54204          172.66379              1569454198   
95474         -43.54204          172.66379              1569454215   
95475         -43.54204          172.66379              1569454300   
94519        -43.539642          172.65609              1569462101   

             Date Time of Day Day of Week         Time Zone fid  \

## 2) Remove Duplicated GPS Entries

In [3]:
a = len(test_data_gdf)

# Boolean mask: rows that are duplicates
mask_keep = ~test_data_gdf.duplicated(subset=['Hashed Device ID','Unix Timestamp of Visit'])
#, 'Latitude of Visit', 'Longitude of Visit'
# Save removed rows
removed_duplicates = test_data_gdf[~mask_keep]

# Apply the filter
test_data_gdf = test_data_gdf[mask_keep]

print(f'Dropped Duplicates ({a - len(test_data_gdf)} rows dropped)')
print("Removed duplicate rows preview:")
print(removed_duplicates.head())

Dropped Duplicates (799 rows dropped)
Removed duplicate rows preview:
        field_1 Polygon ID                          Hashed Device ID  \
112708  1817597        NZL  001830328fe585cef3a4940bfa1490cd83c3878b   
125146  1845613        NZL  012b5efa4bac4d7d96b13631a1aa3a4844467270   
124319  1795445        NZL  012b5efa4bac4d7d96b13631a1aa3a4844467270   
122705  1739033        NZL  0154b443b345eb1cb625aa78ba86c0ab6ca23316   
112303  1747752        NZL  01daf7ad6862ceb90bc79c02f6842798db4a2349   

       Latitude of Visit Longitude of Visit Unix Timestamp of Visit  \
112708        -43.540126         172.657802              1574888419   
125146         -43.54642         172.676742              1573584195   
124319        -43.538376         172.650966              1573584581   
122705        -43.538897         172.653204              1573158475   
112303        -43.539972         172.656815              1574815860   

              Date Time of Day Day of Week         Time Zone fid  \
11

## 3) Remove Rows with NaN in key columns 

In [4]:
b = len(test_data_gdf)

mask_remove = test_data_gdf[['Hashed Device ID','Latitude of Visit','Longitude of Visit','Date','Time of Day']].isna().any(axis=1)
removed_nans = test_data_gdf[mask_remove]

# Apply the filter
test_data_gdf = test_data_gdf[~mask_remove]

print(f'Dropped NaN ({b - len(test_data_gdf)} rows dropped)')
print("Removed NaN rows preview:")
print(removed_nans.head())
print(test_data_gdf['Hashed Device ID'].nunique())

Dropped NaN (1 rows dropped)
Removed NaN rows preview:
        field_1 Polygon ID                          Hashed Device ID  \
132100  1665012        NZL  20f650a7d8d900d1055c456139b20f5f7fee3766   

       Latitude of Visit Longitude of Visit Unix Timestamp of Visit  Date  \
132100        -43.538537         172.651708                          None   

       Time of Day Day of Week Time Zone   fid                     geometry  \
132100        None        None      None  None  POINT (172.65171 -43.53854)   

         t  
132100 NaT  
6581


## 4) Find time in seconds between GPS data points

In [5]:
test_data_gdf["dt"] = test_data_gdf.groupby("Hashed Device ID")["t"].diff().dt.total_seconds()
print(f'Found time between GPS data points')

Found time between GPS data points


## 5) Find distance in meters between GPS data points

In [6]:
start1 = time.time()

distances = [np.nan]  # first row has no previous row

# Iterate through dataframe rows
for i in range(1, len(test_data_gdf)):
    curr_id = test_data_gdf.iloc[i]["Hashed Device ID"]
    prev_id = test_data_gdf.iloc[i-1]["Hashed Device ID"]

    # If the user ID changed → new trajectory → distance = NaN
    if curr_id != prev_id:
        distances.append(np.nan)
        continue

    # Otherwise compute geodesic distance
    lat_prev = test_data_gdf.iloc[i-1]["Latitude of Visit"]
    lon_prev = test_data_gdf.iloc[i-1]["Longitude of Visit"]
    lat_curr = test_data_gdf.iloc[i]["Latitude of Visit"]
    lon_curr = test_data_gdf.iloc[i]["Longitude of Visit"]

    d = geodesic((lat_prev, lon_prev), (lat_curr, lon_curr)).meters
    distances.append(d)

# Assign to DataFrame
test_data_gdf["dist_m"] = distances
print(f'Found distance between GPS data points')

end1 = time.time()
print(f'Code took {end1 - start1:.2f} seconds to find distance between GPS points')

Found distance between GPS data points
Code took 105.95 seconds to find distance between GPS points


# 6) Find speed in meters per second 

In [7]:
test_data_gdf["speed"] = test_data_gdf["dist_m"] / test_data_gdf["dt"]
print(f'Found velocity between data points')

Found velocity between data points


## 7) Find acceleration 

In [8]:
test_data_gdf["accel"] = test_data_gdf.groupby("Hashed Device ID")["speed"].diff() / test_data_gdf["dt"]
print(f'Found acceleration between data points')
print(test_data_gdf['Hashed Device ID'].nunique())

Found acceleration between data points
6581


## 8) Remove GPS data points with impossible values

In [9]:
mask_keep = (
    ((test_data_gdf["speed"].between(0, 40)) | test_data_gdf["speed"].isna()) &
    ((test_data_gdf["accel"].between(-10, 10)) | test_data_gdf["accel"].isna()) &
    ((test_data_gdf["dt"] >= 0) | test_data_gdf["dt"].isna())
)

# Rows that will be removed
removed_rows = test_data_gdf[~mask_keep]

# Apply the filter
test_data_gdf = test_data_gdf[mask_keep]

print(f'{a-len(test_data_gdf)} data points dropped due to impossible values')
# Now you can inspect removed rows
print(removed_rows.head())

print(test_data_gdf['Hashed Device ID'].nunique())

1892 data points dropped due to impossible values
         field_1 Polygon ID                          Hashed Device ID  \
122694   1739022        NZL  0154b443b345eb1cb625aa78ba86c0ab6ca23316   
55707     654757        NZL  021b3b87f65d592de2d867cf0e6b5ff863288c0c   
113139   1848760        NZL  026a6b1b93f6b64857089f656af0699c3e630750   
6812    42584534        NZL  02dcf45713b80ff9524f38eb8cb517f1e995df18   
28067     415933        NZL  04de23f3bfea6d05bb3e3ef9b7c319a730870a85   

       Latitude of Visit Longitude of Visit Unix Timestamp of Visit  \
122694        -43.538815         172.652782              1573158455   
55707         -43.541748         172.663043              1558053336   
113139        -43.549103         172.682573              1574836853   
6812          -43.548833         172.681995              1547533907   
28067         -43.548093         172.680568              1551514650   

              Date Time of Day Day of Week         Time Zone fid  \
122694  2019/11/

## 9) Find trajectories of GPS data

In [10]:
def filter_trajectories(trajectories, min_traj_size, gap_length):
    #print(f'Found trajectories in GPS data')
    
    traj_split = mpd.ObservationGapSplitter(trajectories).split(
        gap=timedelta(minutes=gap_length))
    #print(f'Split trajectories based on large gaps in GPS data')
    
    # Keep only trajectories with >= min_size points
    traj_split.trajectories = [traj for traj in traj_split if traj.size() >= min_traj_size]
    #print(f'Filtered trajectories with little data')
    
    remaining_points = sum(traj.size() for traj in traj_split.trajectories)
    
    print(f"Number of trajectories with min trajectory = {min_traj_size} and min gap = {gap_length} minutes:", len(traj_split))
    #print("Number of GPS data points:", remaining_points)

    return traj_split

#traj_split = filter_trajectories(trajectories = test_traj, min_traj_size = 3, gap_length = 30)

## 10) Filter GPS data on trajectory data quality

In [11]:
def filter_gps_for_trajectories(quality_traj):
    filtered_rows = []
    user_traj_counter = {}
    
    for traj in quality_traj.trajectories:
        # Strip suffix just in case the traj.id also has it
        user_id = traj.id
        user_id = user_id.split('_')[0]  # simpler alternative if all you need is before '_'
        
        df = traj.df.copy()
        
        # Increment counter for this user
        if user_id not in user_traj_counter:
            user_traj_counter[user_id] = 1
        else:
            user_traj_counter[user_id] += 1
        
        df['traj_id'] = user_traj_counter[user_id]
        df['Hashed Device ID'] = user_id  # make sure column is clean
        
        filtered_rows.append(df)
    
    # Concatenate into final GeoDataFrame
    filtered_gdf = gpd.GeoDataFrame(
        pd.concat(filtered_rows, ignore_index=True),
        crs=test_data_gdf.crs
    )
    
    traj_lengths = (
        filtered_gdf
        .groupby(['Hashed Device ID', 'traj_id'])
        .size()  # counts the number of rows per group
        .reset_index(name='traj_length')  # gives a column 'traj_length'
    )
    
    median_length = traj_lengths['traj_length'].median()
    mean_length = traj_lengths['traj_length'].mean()
    
    #print(filtered_gdf.head(n=5))
    #print(filtered_gdf.tail(n=5))

    #print("Median trajectory length:", median_length)
    #print("Mean trajectory length:", mean_length)
    percent_gps = round(100 * len(filtered_gdf)/len(test_data_gdf), 2)    
    #print(f'Filtered GPS data contains {len(filtered_gdf)} data points')
    #print(f'This is {100*len(filtered_gdf)/a:.2f}% of raw GPS data')
    
    #print(f"Filtered GPS data has {filtered_gdf['Hashed Device ID'].nunique()} unique hashed device IDs")
    #print(f"Filtered GPS data has {filtered_gdf['traj_id'].nunique()} unique trajectories")

    return filtered_gdf, percent_gps, median_length, mean_length

#filtered_gdf, percent_gps, median_length, mean_length = filter_gps_for_trajectories(quality_traj = traj_split)

## 11) Save cleaned GPS data to file

In [12]:
filtered_gdf.to_csv(f'Select_GPS_Preprocessing_Params/2GPS_data_cleaned.csv')

end = time.time()
print(f'Code took {end-start:.2f} seconds to clean GPS data')

NameError: name 'filtered_gdf' is not defined

## 12) Selecting Gap Length for Stop Detection and Minimum Trajectory Length for Full GPS Code

In [ ]:
start2 = time.time()

gap_length = [2, 5, 7, 10, 15, 30, 60]
min_traj = [3, 4, 5, 6, 8, 10, 15]

# Initialize empty list to collect results
results = []
iteration = 1

# Calculate trajectories outside of the for loops to avoid repeating loads
test_traj = mpd.TrajectoryCollection(
        test_data_gdf,
        traj_id_col='Hashed Device ID',
        t='t')

for traj in min_traj:
    for gap in gap_length:
        print(f'Loop number {iteration}')
        iteration += 1
        
        # Run your filtering functions
        traj_split = filter_trajectories(trajectories = test_traj, min_traj_size=traj, gap_length=gap)
        filtered_gdf, percent_gps, median_length, mean_length = filter_gps_for_trajectories(quality_traj=traj_split)
        
        # Append results as a dictionary
        results.append({
            'min_traj': traj,
            'gap_length': gap,
            'percentage_gps': percent_gps,
            'median_length': median_length,
            'mean_length': mean_length
        })

# Convert results list to a DataFrame
percent_matrix = pd.DataFrame(results)

### Visualise the results

Want to maximise the remaining GPS data and while balancing a large minimum trajectory size and small gap length

Based on heatmap below a minimum gap length of 10 minutes with minimum trajectory length of 6 looks like a good balance

In [ ]:
plt.figure(figsize=(8, 6))
heatmap_data = percent_matrix.pivot_table(index="min_traj", columns="gap_length",
                             values="percentage_gps")
sns.heatmap(
    heatmap_data,
    annot=True,        # show the percentage in each cell
    fmt=".2f",         # format numbers with 1 decimal place
    cmap="YlGnBu",     # color map
    cbar_kws={'label': 'Percentage of GPS points retained'}
)

plt.title("Percentage of GPS points retained for different parameters")
plt.xlabel("Gap Length (minutes)")
plt.ylabel("Minimum trajectory size")
plt.yticks(rotation=0)  # keep y-axis labels horizontal

plt.savefig(f'Select_GPS_Preprocessing_Params/2Heatmap_conserved_gps_data_by_var.png')

plt.show()

percent_matrix.to_csv(f'Select_GPS_Preprocessing_Params/2Percantage_gps_after_preprocessing.csv')

In [ ]:
# Optional: use a clean seaborn style
sns.set(style="whitegrid")

# Metrics you want to plot
metrics = ['percentage_gps', 'mean_length', 'median_length']
titles = ['Percentage of GPS Points', 'Mean Trajectory Length', 'Median Trajectory Length']

# Create subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True)

# Loop through metrics and axes
for ax, metric, title in zip(axes, metrics, titles):
    # Plot each min_traj as a separate line
    for mt in percent_matrix['min_traj'].unique():
        sub = percent_matrix[percent_matrix['min_traj'] == mt]
        ax.plot(sub['gap_length'], sub[metric], marker='o', label=f"min_gap={mt}")

    ax.set_title(title)
    ax.set_xlabel('Gap Length')
    ax.set_ylabel(metric)
    if metric == 'percentage_gps':
        ax.legend(title="Min Traj")

plt.tight_layout()

plt.savefig(f'Select_GPS_Preprocessing_Params/2Min_gap_legnth_vs_params.png')

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# --- Create pivot for surface ---
pivot = percent_matrix.pivot_table(
    index='gap_length',
    columns='min_traj',
    values='percentage_gps'
)

X, Y = np.meshgrid(pivot.columns, pivot.index)
Z = pivot.values

fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')

# --- Plot semi-transparent surface ---
ax.plot_surface(
    X, Y, Z,
    alpha=0.4,      # transparency (0 = invisible, 1 = opaque)
    cmap = 'viridis',
    edgecolor='white'
)

# --- Plot the actual points ---
ax.scatter(
    percent_matrix['min_traj'],
    percent_matrix['gap_length'],
    percent_matrix['percentage_gps'],
    s=40,
    alpha=1
)

ax.set_ylabel('Gap Length')
ax.set_xlabel('Min Trajectory Size')
ax.set_zlabel('Percentage GPS')
ax.zaxis.labelpad=-0.7

ax.view_init(elev=20)

plt.savefig(f'Select_GPS_Preprocessing_Params/23D_Plot_Min_Traj_Min_Gap_Length_Percent_GPS.png')

plt.show()


In [ ]:
end2 = time.time()
print(f'Code took {end2-start2:.2f} seconds to select GPS cleaning parameter values')

# Load csv in chunks

##### Define the path to your large CSV file
file_path = 'large_data.csv'

##### Define the chunk size (e.g., 10,000 rows per chunk)
chunk_size = 10000

###### Initialize an empty list to store processed chunks or aggregated results
processed_results = []

###### Iterate over the CSV file in chunks
for chunk in pd.read_csv(file_path, chunksize=chunk_size):
    